## 1、静态模型

#### 静态模型其实就是只用一个LLM完成整个推理过程

In [2]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

## 2、动态模型

#### 动态模型其实就是你自己设条件，然后根据条件选择LLM进行推理

In [3]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatOpenAI(model="gpt-4o-mini")        #节省成本
advanced_model = ChatOpenAI(model="gpt-4o")          #性能更强

@wrap_model_call                                     #中间件
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    """基于条件选择模型"""
    message_count = len(request.state["messages"])   #对话轮数

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advanced_model
    else:
        model = basic_model

    request.model = model
    return handler(request)

agent = create_agent(
    model=basic_model,  # Default model
    middleware=[dynamic_model_selection]
)